# 01 · OpenAI-compatible 채팅

이 노트북은 루트 .env의 LLM_MODEL(기본값: gpt-5.6-luna)과 SAP AI Core service key로 모델을 호출합니다. 이어서 LiteLLM Proxy URL과 API key가 설정된 경우에는 LITELLM_MODEL(없으면 LLM_MODEL)로 같은 메시지를 공식 OpenAI SDK 형식으로 호출합니다. 비밀값은 출력하지 않습니다.

## 1. 공통 코드 불러오기

프로젝트 루트의 .env를 읽고, 표시 가능한 상태 정보만 확인합니다.

In [ ]:
from pathlib import Path
import sys

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '1_gen_ai_hub' / 'src').is_dir())
sys.path[:0] = [str(project_root / '1_gen_ai_hub' / 'src'), str(project_root / '1_gen_ai_hub')]

from lab_common.llm import chat_text
from lab_common.config import load_settings
from src.environment import environment_status
from src.litellm_adapter import chat_completion, litellm_connection_status

settings = load_settings()
environment_status(settings)

## 2. SAP AI Core로 LLM_MODEL 호출

SAP Gen AI Hub SDK가 service key를 OAuth access token으로 교환한 뒤 .env의 LLM_MODEL을 호출합니다. 이 경로는 현재 .env의 AI Core 설정으로 검증되었습니다.

In [ ]:
model = settings.llm_model
messages = [
    {'role': 'system', 'content': '너는 간결한 한국어 교육 도우미다.'},
    {'role': 'user', 'content': '응답 형식 테스트입니다. 정확히 OK만 답해줘.'},
]

reply = chat_text(settings, messages, model=model)
{'model': model, 'reply': reply}

## 3. 선택: LiteLLM Proxy를 OpenAI SDK로 호출

LiteLLM Proxy는 별도의 OpenAI-compatible gateway입니다. 아래 셀은 LITELLM_BASE_URL, LITELLM_API_KEY, LITELLM_MODEL이 모두 설정된 경우에만 외부 호출을 합니다. AI_SECRET_KEY도 API key의 호환 별칭으로 지원합니다.

예시 설정:
LITELLM_BASE_URL=https://your-litellm.example/v1
LITELLM_API_KEY=your-ai-secret-key
LITELLM_MODEL=gpt-5.6-luna

In [ ]:
proxy_status = litellm_connection_status()
proxy_status

In [ ]:
if proxy_status['litellm_ready']:
    proxy_reply = chat_completion(messages)
    {'model': proxy_status['model'], 'reply': proxy_reply}
else:
    print('LiteLLM Proxy 설정이 없어 호출하지 않았습니다.')
    print('누락 설정:', ', '.join(proxy_status['missing_litellm_fields']))

## 다음 단계

SAP AI Core service key와 LiteLLM API key는 서로 다른 인증 방식입니다. AI Core 모델 호출에는 2번 셀을 사용하고, 조직의 LiteLLM Proxy가 제공되는 경우에만 3번 셀의 OpenAI-compatible adapter를 사용하세요.